In [ ]:
from google.colab import drive
import os
import glob

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define the exact Drive paths (Fixed Capitalization!)
DRIVE_ROOT = '/content/drive/MyDrive/COROnet_Drive'
INPUT_DIR = os.path.join(DRIVE_ROOT, "ICC_Data")
OUTPUT_ROOT = os.path.join(DRIVE_ROOT, "Segmentation/Helper/Totalsegmentor_Heart")

# 3. Create the output directory if it doesn't exist yet
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# 4. Scan for the input files
input_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.nii.gz")))
print(f"✅ Found {len(input_files)} raw CT scans ready for chambers extraction.")

In [ ]:
print("Installing TotalSegmentator and dependencies...")
!pip install TotalSegmentator nibabel numpy -q

import torch
import gc
import shutil
from totalsegmentator.python_api import totalsegmentator

# Input your academic license key here (Required for V2)
!totalseg_set_license -l aca_PBRE81JH1TV8M3

# Function to wipe GPU memory between heavy patients
def clear_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.synchronize()

print("✅ Environment setup complete. GPU memory cleared.")
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

In [ ]:
LOCAL_TEMP = "/content/temp_out"

for i, input_path in enumerate(input_files):
    filename = os.path.basename(input_path)
    patient_id = filename.replace("_0000.nii.gz", "").replace(".nii.gz", "")
    patient_drive_dir = os.path.join(OUTPUT_ROOT, patient_id)

    # Resume capability: Skip if the heart file already exists on Drive
    drive_heart_file = os.path.join(patient_drive_dir, "heart.nii.gz")
    if os.path.exists(drive_heart_file) and os.path.getsize(drive_heart_file) > 1024:
        print(f"[{i+1}/{len(input_files)}] ⏭️ Skipping {patient_id} (Already exists)")
        continue

    print(f"\n{'='*50}\n[{i+1}/{len(input_files)}] ⚙️ PROCESSING: {patient_id}\n{'='*50}")

    # Prepare directories
    os.makedirs(patient_drive_dir, exist_ok=True)
    if os.path.exists(LOCAL_TEMP): shutil.rmtree(LOCAL_TEMP)
    os.makedirs(LOCAL_TEMP, exist_ok=True)

    # Force output to a single file path in the local temp folder
    local_heart_file = os.path.join(LOCAL_TEMP, "heart.nii.gz")

    try:
        clear_gpu_memory()

        print(f"-> Extracting Chambers (High-Res) to {local_heart_file}...")
        # Using -ta heartchambers_highres with --ml and a specific file path
        !TotalSegmentator -i "{input_path}" -o "{local_heart_file}" -ta heartchambers_highres --ml -d gpu

        # --- SHIPPING PHASE ---
        if os.path.exists(local_heart_file):
            print(f"-> Shipping heart.nii.gz to Google Drive...")
            shutil.copy(local_heart_file, drive_heart_file)

            # Verify the copy before removing local
            if os.path.exists(drive_heart_file) and os.path.getsize(drive_heart_file) > 1024:
                os.remove(local_heart_file)
                print(f"✓ {patient_id} processed and saved successfully.")
            else:
                print(f"🚨 Upload failed for {patient_id}. Keeping local copy.")
        else:
            # Check if it created a folder instead of a file (some versions do this)
            if os.path.isdir(local_heart_file):
                print("-> Moving directory contents...")
                for f in os.listdir(local_heart_file):
                    shutil.copy(os.path.join(local_heart_file, f), os.path.join(patient_drive_dir, f))
                print(f"✓ {patient_id} (directory) processed successfully.")
            else:
                raise Exception(f"Output file {local_heart_file} was not found.")

    except Exception as e:
        print(f"🚨 FAILED {patient_id}: {e}")

print("\n\n✅ ALL PATIENTS PROCESSED!")